# RAG

In [1]:
# Load the environment
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# Read the model name
import os
MODEL_NAME = os.environ["GEMINI_MODEL"]
API_KEY = os.environ["GOOGLE_GENERATIVE_AI_API_KEY"]

In [3]:
# Create the LangChain model
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI (model = MODEL_NAME, api_key =API_KEY, temperature = 0)

In [4]:
# Create the document object
from langchain_core.documents import Document
documents = [
    Document(
        page_content="LangChain provides abstractions for building LLM applications.",
        metadata={"source": "langchain.txt"}
    ),
    Document(
        page_content="LangGraph is designed for stateful agent workflows.",
        metadata={"source": "langgraph.txt"}
    ),
]

## Manual RAG

In [5]:
# Create the embedding model
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=API_KEY,
)

In [6]:
# Create the vector store
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embedding=embeddings)

In [7]:
# Add documents to the vector_store

vector_store.add_documents(documents)

['0a840515-b72a-42c9-871e-9e1cff7b51eb',
 'f1da3468-717c-41e4-893e-0b5d2bd188ab']

In [8]:
# Run the retriever

retriever = vector_store.as_retriever(
    search_kwargs = {"k":2}
)

In [9]:
# Invoke the retriever

query = "What is LangGraph"
retrieved_docs = retriever.invoke(query)

In [ ]:
# Create the formatting function
def format_docs(documents: list[Document])->str:
    return "\n\n".join(
        doc.page_content
        for doc in documents
    )

"""
or
def docs_to_text(documents: list[Document])-> str:
    text = ""
    for doc in documents:
        text = text + doc.page_content + "\n\n"
    return text
"""

In [63]:
context = format_docs(retrieved_docs)
print(context)

LangGraph is designed for stateful agent workflows.

LangChain provides abstractions for building LLM applications.


In [12]:
# Create the prompt

prompt = f"""
Answer the question using the following context.

Context:
{context}

Question:
{query}
"""

In [13]:
# Invoke the llm

response = llm.invoke(prompt)

print(response.content)

[{'type': 'text', 'text': 'Based on the provided context, LangGraph is designed for stateful agent workflows.', 'extras': {'signature': 'EoAGCv0FARFNMg+nDCfiR8NfQtbL6vY2fdrneKqE1Nc931MiGGDeViQnWUlPXbS8ug2SkV1eAoL/h7Xg7S5mIF0HEXFfeN6yZj6l0V/9YR5VX3kzkZYZZEcVXLJJA9wBVyVg6E6/5SF7huMZidliORfq4lD05NpmAxQIFUnXRNUTmMpuDW6PdXSOCCDaTL34nvaVblE5hfjhvFq2GOfI9B7TNanDL8ic2NWLgYToCQbxgDb3L7EGsr/oyhRdr3ttQM3VCnix1OyjERoE6sI0CIfw9UtXlOR9unMPqrYsDSyGji3f4Q0ZAvgdpEkKQxluswz53SsjXWAzb5/lZwQGHWVUJQtJwRYaKRnFQQZ7yOge6YbxqtDBqxs4AVmyaoTZjpfAc2C36cDrWtssSiARiU6XFCfCuvfI1QsWNID6auRfqh9Zv8rOQik0rp0egtXG2A6tmw8EY4TCc/EoO6pueDImszbZ5C4GHnASMrwKd4wEw1UJbNpgnijEbmSvlv+73z3LKlGGOrsn+SDtVBzLCWQrKXcGf2UTCAQwaJH9iVgRqLFGaD0+vAiaVFrsNoZy0Dor2xYhQF5HQDn1ZxDqWyPOY7O6Aj/3DfjxazHNKHJjwPahWx7+jep4E+ZN9VvdBCClWhjyJ2gEsUjGyS/U/E6i57HKme7H8OHwITKR/4uMa9BB6qlTXF9LGPZv7mhnAqeJBzt3WUTgRh6TBNzNsIKJgvTx6oyH06N9gqg3+prvuv+MokZScAfvNVMUO0Gc5RwPrPDXhb3qPavNlALs7ItYAjFev2jgnkX41lmmUDhEhYOuJkxCWSd38n+xK/9M/qbzNjKSf7WETOeUHrD2Ifero8cjU+s